# 🎭 ChuckleNet: HYBRID Fast Pipeline

**Strategy**:
1. **Text-only** (instant) → all 620 videos → text baseline
2. **F0 only** (15 min) → sample 30 videos → verify audio adds value
3. **Combine** → best of both worlds

**Runtime**: ~20-30 min total

In [ ]:
# 1. Setup
!apt-get install -y ffmpeg 2>&1 | tail -1
!pip install librosa numpy pandas scikit-learn tqdm transformers 2>&1 | tail -3

from google.colab import drive
drive.mount('/content/drive')

import os, glob, re, time
import numpy as np
import pandas as pd
import subprocess
from tqdm import tqdm

for BASE in ['/content/drive/My Drive/chuckle_net', '/content/drive/Shareddrives/chuckle_net']:
    if os.path.exists(BASE): break

AUDIO_DIR = f'{BASE}/audio'
VTT_DIR = f'{BASE}/vtt'

audio_files = glob.glob(f'{AUDIO_DIR}/*.m4a') + glob.glob(f'{AUDIO_DIR}/*.wav')
vtt_files = glob.glob(f'{VTT_DIR}/*.vtt')

print(f'Audio: {len(audio_files)}, VTT: {len(vtt_files)}')

In [ ]:
# 2. Parse ALL VTT → Text Features + Labels (INSTANT)
def get_vid(name):
    return name.replace('.en.vtt','').replace('.vtt','').replace('.m4a','').replace('.wav','')

def parse_vtt(vtt_path):
    """Return list of (start, end, text, has_laughter)."""
    with open(vtt_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    
    def to_sec(ts):
        p = ts.replace('.',':').split(':')
        return int(p[0])*3600 + int(p[1])*60 + float(p[2])
    
    cues, lines = [], content.split('\n')
    i = 0
    while i < len(lines):
        if '-->' in lines[i]:
            s, e = lines[i].split('-->')
            s, e = to_sec(s.strip()), to_sec(e.strip())
            txt, i = [], i+1
            while i < len(lines) and lines[i].strip() and '-->' not in lines[i]:
                txt.append(lines[i].strip()); i += 1
            cues.append((s, e, ' '.join(txt), '[laughter]' in ' '.join(txt).lower()))
        else: i += 1
    return cues

def text_to_features(text, max_len=50):
    """Simple TF-IDF-like text features (fast, no GPU needed)."""
    # Character count features
    text_lower = text.lower()
    return [
        len(text),  # length
        text_lower.count('.'),  # sentences
        text_lower.count(' '),  # words  
        sum(1 for c in text if c.isupper()),  # caps
        text_lower.count('haha'),  # laughter text
        text_lower.count('laugh'),
        text_lower.count('oh'),
        text_lower.count('yeah'),
        text_lower.count('okay'),
        text_lower.count('right'),
    ]

# Parse all VTT
print('Parsing VTT files (instant)...')
t0 = time.time()

all_text = []
all_feat = []
all_labels = []
all_vids = []
all_langs = []

vtt_lookup = {get_vid(os.path.basename(v)): v for v in vtt_files}
audio_lookup = {get_vid(os.path.basename(a)): a for a in audio_files}
matching = set(audio_lookup.keys()) & set(vtt_lookup.keys())

for vid in tqdm(matching, desc='Parsing'):
    vtt_path = vtt_lookup[vid]
    
    lang = 'en'
    if '.hi.' in vtt_path: lang = 'hi'
    elif '.zh.' in vtt_path: lang = 'zh'
    elif '.es.' in vtt_path: lang = 'es'
    
    cues = parse_vtt(vtt_path)
    
    for i, (s, e, text, has_laugh) in enumerate(cues):
        if e > s and e - s < 30:  # Valid utterance
            all_text.append(text)
            all_feat.append(text_to_features(text))
            all_labels.append(1 if has_laugh else 0)
            all_vids.append(vid)
            all_langs.append(lang)

t1 = time.time()
print(f'\n✅ Parsed {len(all_text)} utterances in {t1-t0:.1f}s')
print(f'Positive: {sum(all_labels)} ({100*np.mean(all_labels):.1f}%)')

X_text = np.array(all_feat)
y = np.array(all_labels)
vids = np.array(all_vids)
langs = np.array(all_langs)

for lang in ['en', 'hi', 'zh', 'es']:
    mask = langs == lang
    if mask.sum() > 0:
        print(f'  {lang}: {mask.sum()} segs, {100*y[mask].mean():.1f}% positive')

In [ ]:
# 3. Train TEXT-ONLY Baseline
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score, precision_score, recall_score

unique_vids = list(set(vids))
np.random.seed(42); np.random.shuffle(unique_vids)
n_test = max(1, int(len(unique_vids)*0.2))
test_vids = set(unique_vids[:n_test])

train_mask = ~np.isin(vids, list(test_vids))
X_train, X_test = X_text[train_mask], X_text[~train_mask]
y_train, y_test = y[train_mask], y[~train_mask]

print(f'Train: {len(X_train)} ({100*y_train.mean():.1f}% pos)')
print(f'Test: {len(X_test)} ({100*y_test.mean():.1f}% pos)\n')

lr = LogisticRegression(max_iter=1000, C=1.0)
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)
print('=== TEXT-ONLY Logistic Regression ===')
print(f'F1: {f1_score(y_test, y_pred):.4f}')
print(f'Precision: {precision_score(y_test, y_pred):.4f}')
print(f'Recall: {recall_score(y_test, y_pred):.4f}')

mlp = MLPClassifier(hidden_layer_sizes=(64,32), max_iter=500, random_state=42)
mlp.fit(X_train, y_train)
y_pred_mlp = mlp.predict(X_test)
print('\n=== TEXT-ONLY MLP ===')
print(f'F1: {f1_score(y_test, y_pred_mlp):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_mlp):.4f}')
print(f'Recall: {recall_score(y_test, y_pred_mlp):.4f}')

In [ ]:
# 4. Extract F0 for SAMPLE 30 videos (~15 min)
import tempfile
import librosa

def extract_f0(audio_path, start, end, sr=22050):
    duration = end - start
    if duration <= 0 or duration > 30: return None
    with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as tmp:
        tmp_path = tmp.name
    try:
        cmd = ['ffmpeg', '-y', '-ss', str(start), '-t', str(duration),
               '-i', audio_path, '-ar', str(sr), '-ac', '1', '-loglevel', 'error', tmp_path]
        if subprocess.run(cmd, timeout=10).returncode != 0: return None
        y, _ = librosa.load(tmp_path, sr=sr)
        if len(y) < sr * 0.05: return None
        f0, _, _ = librosa.pyin(y, fmin=50, fmax=500, sr=sr, hop_length=512)
        f0 = np.nan_to_num(f0, nan=0.0)
        return [float(np.mean(f0)), float(np.std(f0)), float(np.max(f0)),
                float(np.min(f0)), float(np.mean(f0>0))]
    except: return None
    finally:
        if os.path.exists(tmp_path): os.unlink(tmp_path)

# Sample 30 diverse videos
unique_vids_list = list(set(vids))
np.random.seed(42)
sample_vids = np.random.choice(unique_vids_list, size=min(30, len(unique_vids_list)), replace=False)
sample_vids = set(sample_vids)

print(f'Extracting F0 from {len(sample_vids)} sample videos...')
print(f'Estimated time: ~15 min\n')

f0_data = []
for vid in tqdm(sample_vids, desc='F0 extraction'):
    audio_path = audio_lookup.get(vid)
    vtt_path = vtt_lookup.get(vid)
    if not audio_path or not vtt_path: continue
    
    cues = parse_vtt(vtt_path)
    lang = 'en'
    if '.hi.' in vtt_path: lang = 'hi'
    elif '.zh.' in vtt_path: lang = 'zh'
    elif '.es.' in vtt_path: lang = 'es'
    
    for i, (s, e, text, has_laugh) in enumerate(cues):
        feat = extract_f0(audio_path, s, e)
        if feat:
            f0_data.append({
                'vid': vid, 'uid': f'{vid}_{i}',
                'f0': feat, 'label': 1 if has_laugh else 0,
                'lang': lang, 'text': text
            })

f0_df = pd.DataFrame(f0_data)
print(f'\n✅ F0 extracted: {len(f0_df)} segments from {len(sample_vids)} videos')
print(f'Positive: {f0_df["label"].sum()} ({100*f0_df["label"].mean():.1f}%)')

In [ ]:
# 5. Train F0 Model on Sample
X_f0 = np.array(f0_df['f0'].tolist())
y_f0 = np.array(f0_df['label'])

unique_f0_vids = list(set(f0_df['vid']))
np.random.seed(42); np.random.shuffle(unique_f0_vids)
n_test = max(1, int(len(unique_f0_vids)*0.2))
f0_test_vids = set(unique_f0_vids[:n_test])

f0_train_mask = ~np.isin(f0_df['vid'], list(f0_test_vids))
X_f0_train, X_f0_test = X_f0[f0_train_mask], X_f0[~f0_train_mask]
y_f0_train, y_f0_test = y_f0[f0_train_mask], y_f0[~f0_train_mask]

print(f'F0 Train: {len(X_f0_train)} ({100*y_f0_train.mean():.1f}% pos)')
print(f'F0 Test: {len(X_f0_test)} ({100*y_f0_test.mean():.1f}% pos)\n')

lr_f0 = LogisticRegression(max_iter=1000, C=1.0)
lr_f0.fit(X_f0_train, y_f0_train)
y_f0_pred = lr_f0.predict(X_f0_test)
print('=== F0 Logistic Regression ===')
print(f'F1: {f1_score(y_f0_test, y_f0_pred):.4f}')
print(f'Precision: {precision_score(y_f0_test, y_f0_pred):.4f}')
print(f'Recall: {recall_score(y_f0_test, y_f0_pred):.4f}')

In [ ]:
# 6. Save
import pickle

# Save text data
out = {
    'text_features': X_text,
    'labels': y,
    'vids': vids,
    'langs': langs,
    'all_text': all_text
}
np.savez_compressed(f'{BASE}/text_features_620.npz', **out)

# Save F0 sample
f0_df.to_pickle(f'{BASE}/f0_sample_30.pkl')

# Save models
with open(f'{BASE}/text_model.pkl', 'wb') as f:
    pickle.dump(lr, f)
with open(f'{BASE}/f0_model.pkl', 'wb') as f:
    pickle.dump(lr_f0, f)

print(f'Saved to {BASE}/')
print('  text_features_620.npz')
print('  f0_sample_30.pkl')
print('  text_model.pkl')
print('  f0_model.pkl')

print('\n🎉 PIPELINE COMPLETE!')